# ⚡ Colab D — PyTorch Lightning: High-Level Training
**LightningModule + LightningDataModule + Trainer**

In [ ]:
!pip install pytorch-lightning -q
import pytorch_lightning as pl, torch, torch.nn as nn
import numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset, random_split
torch.manual_seed(42); np.random.seed(42)
print('PL version:', pl.__version__)

## 📊 Section 1 — Data

In [ ]:
N=1000
x1=np.random.uniform(-2,2,(N,1)); x2=np.random.uniform(-2,2,(N,1)); x3=np.random.uniform(-2,2,(N,1))
y=2*x1**2+3*x2*x3+np.sin(x1*x2)+0.5*x3**2+np.random.normal(0,0.1,(N,1))
X=np.hstack([x1,x2,x3])
X_n=(X-X.mean(0))/X.std(0); y_mean,y_std=y.mean(),y.std(); y_n=(y-y_mean)/y_std

## 🗂️ Section 2 — LightningDataModule

In [ ]:
class RegressionDM(pl.LightningDataModule):
    def __init__(self, X, y, batch_size=64, val_frac=0.2):
        super().__init__()
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.batch_size, self.val_frac = batch_size, val_frac

    def setup(self, stage=None):
        ds = TensorDataset(self.X, self.y)
        n_val = int(len(ds)*self.val_frac)
        self.train_ds, self.val_ds = random_split(ds, [len(ds)-n_val, n_val])

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=256)

dm = RegressionDM(X_n, y_n)
print('DataModule created')

## 🏗️ Section 3 — LightningModule

In [ ]:
class NeuralNetLightning(pl.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nn.Linear(3,64), nn.ReLU(),
            nn.Linear(64,32), nn.ReLU(),
            nn.Linear(32,16), nn.Tanh(),
            nn.Linear(16,1)
        )
        # He init
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x): return self.net(x)

    def training_step(self, batch, batch_idx):
        X, y = batch
        loss = nn.functional.mse_loss(self(X), y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        X, y = batch
        loss = nn.functional.mse_loss(self(X), y)
        self.log('val_loss', loss, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt,'min',patience=50,factor=0.5)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sch, 'monitor': 'val_loss'}}

model = NeuralNetLightning(lr=1e-3)
print(model)

## 🏋️ Section 4 — Trainer

In [ ]:
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=80, verbose=True),
    ModelCheckpoint(monitor='val_loss', save_top_k=1, filename='best-{epoch}-{val_loss:.4f}')
]

trainer = pl.Trainer(
    max_epochs=1000,
    callbacks=callbacks,
    log_every_n_steps=10,
    enable_progress_bar=True
)

trainer.fit(model, dm)
print(f'Training finished at epoch {trainer.current_epoch}')

## 📈 Section 5 — Results

In [ ]:
# Extract logged metrics
import pandas as pd
metrics = pd.read_csv(f'{trainer.logger.log_dir}/metrics.csv') if hasattr(trainer.logger,'log_dir') else None

# Predict on test portion
model.eval()
X_te = torch.tensor(X_n[int(0.8*N):], dtype=torch.float32)
y_te = y_n[int(0.8*N):]
with torch.no_grad():
    yp = model(X_te).numpy()*y_std+y_mean
    yt = y_te*y_std+y_mean

plt.figure(figsize=(6,6))
plt.scatter(yt,yp,alpha=0.4,s=15,color='steelblue')
mn,mx=yt.min(),yt.max()
plt.plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect')
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title('Colab D — PyTorch Lightning'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Test MSE: {((yp-yt)**2).mean():.4f}')